In [ ]:
# Step 1: Check for GPU and install drivers
import subprocess
import sys

def check_gpu():
    """Check if GPU is available and print GPU info"""
    print("=" * 60)
    print("GPU Availability Check")
    print("=" * 60)

    # Check using nvidia-smi
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✓ NVIDIA GPU detected!")
            print(result.stdout)
            return True
        else:
            print("✗ nvidia-smi failed")
            print(result.stderr)
            return False
    except FileNotFoundError:
        print("✗ nvidia-smi not found")
        return False

def install_gpu_drivers():
    """Install NVIDIA drivers and CUDA toolkit if needed"""
    print("\n" + "=" * 60)
    print("Installing GPU Drivers")
    print("=" * 60)

    # For Google Colab, GPU drivers are usually pre-installed
    # This checks and installs if needed

    # Install NVIDIA drivers
    print("Installing nvidia-driver... (if not already installed)")
    subprocess.run([sys.executable, "-m", "pip", "install", "nvidia-driver", "-q"],
                   capture_output=True)

    # Install CUDA toolkit
    print("Installing CUDA toolkit...")
    subprocess.run([sys.executable, "-m", "pip", "install", "cuda-python", "-q"],
                   capture_output=True)

    print("Driver installation complete!")

# Check GPU availability
gpu_available = check_gpu()

if gpu_available:
    print("\n✅ GPU is ready for deep learning tasks!")
    install_gpu_drivers()
else:
    print("\n⚠️ No GPU detected. The classifier will run on CPU (slower).")
    print("    For GPU acceleration, ensure you have:")
    print("    - NVIDIA GPU with drivers installed")
    print("    - CUDA toolkit 11.8+ installed")
    print("    - In Google Colab: Runtime > Change runtime type > GPU")

In [ ]:
# Step 2: Install required packages
import subprocess
import sys

print("=" * 60)
print("Installing Required Packages")
print("=" * 60)

packages = [
    "torch>=2.2",
    "scikit-learn>=1.4",
    "numpy>=1.26",
    "fastapi>=0.110",
    "uvicorn[standard]>=0.29",
    "pydantic>=2.0"
]

for package in packages:
    print(f"Installing {package}...")
    subprocess.run([sys.executable, "-m", "pip", "install", package, "-q"],
                   capture_output=True)

print("\n✅ All packages installed!")

In [ ]:
# Step 3: Set up project structure and generate data
import os
import json
import random
from pathlib import Path

print("=" * 60)
print("Generating Synthetic Training Data")
print("=" * 60)

# Labels for classification
LABELS = ["normal", "driver_issue", "hardware_issue", "memory_pressure", "kubernetes_issue", "node_issue"]

def sample(label):
    """Generate synthetic GPU fault data based on label"""
    x = {
        "temperature": random.gauss(55, 8),
        "memory_used_pct": random.gauss(45, 15),
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    }
    if label == "driver_issue":
        x.update(driver_ok=0, cuda_visible=0)
    elif label == "hardware_issue":
        x.update(temperature=random.gauss(88, 5), ecc_errors=random.randint(2, 20), xid_errors=random.randint(1, 8))
    elif label == "memory_pressure":
        x.update(memory_used_pct=random.gauss(94, 3))
    elif label == "kubernetes_issue":
        x.update(pod_ready=0, cuda_visible=0)
    elif label == "node_issue":
        x.update(node_ready=0, pod_ready=0)
    return {"features": x, "label": label}

# Generate data
random.seed(42)
rows = [sample(label) for label in LABELS for _ in range(300)]
random.shuffle(rows)

# Create directories
Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

# Save to JSONL
Path("data/gpu_faults.jsonl").write_text("\n".join(json.dumps(r) for r in rows) + "\n")

print(f"✅ Generated {len(rows)} synthetic GPU fault records")
print(f"   Saved to: data/gpu_faults.jsonl")

# Show label distribution
from collections import Counter
label_counts = Counter(r["label"] for r in rows)
print("\nLabel distribution:")
for label, count in label_counts.items():
    print(f"  - {label}: {count} samples")

In [ ]:
# Step 4: Train the GPU Fault Classifier
import json
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

print("=" * 60)
print("Training GPU Fault Classifier")
print("=" * 60)

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Define features and labels
FEATURES = ["temperature", "memory_used_pct", "ecc_errors", "xid_errors",
            "pod_ready", "driver_ok", "cuda_visible", "node_ready"]
LABELS = ["normal", "driver_issue", "hardware_issue", "memory_pressure",
          "kubernetes_issue", "node_issue"]

# Load data
print("\nLoading training data...")
rows = [json.loads(x) for x in Path("data/gpu_faults.jsonl").read_text().splitlines()]
X = np.array([[r["features"][f] for f in FEATURES] for r in rows], dtype="float32")
y = np.array([LABELS.index(r["label"]) for r in rows])
print(f"Total samples: {len(X)}")

# Split data
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=.3, stratify=y, random_state=42)
Xv, Xte, yv, yte = train_test_split(Xtmp, ytmp, test_size=.5, stratify=ytmp, random_state=42)
print(f"Train: {len(Xtr)}, Validation: {len(Xv)}, Test: {len(Xte)}")

# Scale features
scaler = StandardScaler().fit(Xtr)
Xtr, Xv, Xte = [scaler.transform(x).astype("float32") for x in (Xtr, Xv, Xte)]

# Define model
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Dropout(.15),
            nn.Linear(32, 6)
        )
    def forward(self, x):
        return self.net(x)

# Initialize model
model = Classifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
                    batch_size=64, shuffle=True)

# Training loop
print("\nTraining model...")
for epoch in range(60):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss_fn(model(xb), yb).backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_preds = model(torch.tensor(Xv).to(device)).argmax(1).cpu().numpy()
        val_acc = accuracy_score(yv, val_preds)
        print(f"  Epoch {epoch+1}/60 - Validation Accuracy: {val_acc:.4f}")

# Evaluation
print("\n" + "=" * 60)
print("Model Evaluation on Test Set")
print("=" * 60)
model.eval()
with torch.no_grad():
    Xte_tensor = torch.tensor(Xte).to(device)
    predictions = model(Xte_tensor).argmax(1).cpu().numpy()

print(f"\nAccuracy: {accuracy_score(yte, predictions):.4f}")
print(f"Macro F1 Score: {f1_score(yte, predictions, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(yte, predictions, target_names=LABELS))

# Save model
torch.save({
    "state_dict": model.state_dict(),
    "mean": scaler.mean_,
    "scale": scaler.scale_,
    "features": FEATURES,
    "labels": LABELS
}, "artifacts/model.pt")
print("\n✅ Model saved to artifacts/model.pt")

In [ ]:
# Step 5: Make Predictions on New Data
import torch
import numpy as np
import torch.nn as nn

# Define Classifier inline (train.py not available in Colab)
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Dropout(.15),
            nn.Linear(32, 6)
        )
    def forward(self, x):
        return self.net(x)

print("=" * 60)
print("Making Predictions")
print("=" * 60)

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = torch.load("artifacts/model.pt", map_location=device, weights_only=False)
model = Classifier()
model.load_state_dict(bundle["state_dict"])
model.to(device)
model.eval()

def predict(observation):
    """Predict GPU fault label from observation"""
    features = bundle["features"]
    x = np.array([[observation.get(f, 0) for f in features]], dtype="float32")
    x = ((x - bundle["mean"]) / bundle["scale"]).astype("float32")

    with torch.no_grad():
        probs = torch.softmax(model(torch.tensor(x).to(device)), 1)[0].cpu().numpy()

    order = probs.argsort()[::-1]
    return {
        "label": bundle["labels"][int(order[0])],
        "confidence": float(probs[order[0]]),
        "probabilities": {bundle["labels"][int(i)]: float(probs[i]) for i in order}
    }

# Example observations to test
test_observations = [
    {
        "temperature": 52,
        "memory_used_pct": 42,
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    },  # Normal
    {
        "temperature": 85,
        "memory_used_pct": 45,
        "ecc_errors": 15,
        "xid_errors": 5,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    },  # Hardware issue
    {
        "temperature": 58,
        "memory_used_pct": 95,
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    },  # Memory pressure
    {
        "temperature": 55,
        "memory_used_pct": 40,
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 0,
        "driver_ok": 0,
        "cuda_visible": 0,
        "node_ready": 1
    },  # Driver issue
]

print("\nTest Predictions:")
for i, obs in enumerate(test_observations):
    result = predict(obs)
    print(f"\n[Sample {i+1}] Observed values:")
    print(f"  Temperature: {obs['temperature']}°C")
    print(f"  Memory: {obs['memory_used_pct']}%")
    print(f"  ECC Errors: {obs['ecc_errors']}")
    print(f"  XID Errors: {obs['xid_errors']}")
    print(f"  → Predicted: {result['label']} (confidence: {result['confidence']:.2%})")

In [ ]:
# Step 6: Start FastAPI Server (optional)
# Uncomment below to run the API server

# from fastapi import FastAPI
# from pydantic import BaseModel, Field
# import numpy as np
# import torch
# import torch.nn as nn
#
# class Classifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(8, 32),
#             nn.ReLU(),
#             nn.Dropout(.15),
#             nn.Linear(32, 6)
#         )
#     def forward(self, x):
#         return self.net(x)
#
# app = FastAPI(title="GPU Fault Classifier", version="0.1.0")
# bundle = torch.load("artifacts/model.pt", map_location="cpu", weights_only=False)
# model = Classifier()
# model.load_state_dict(bundle["state_dict"])
# model.eval()
#
# class Observation(BaseModel):
#     temperature: float = Field(55, ge=-20, le=150)
#     memory_used_pct: float = Field(45, ge=0, le=100)
#     ecc_errors: int = Field(0, ge=0)
#     xid_errors: int = Field(0, ge=0)
#     pod_ready: int = Field(1, ge=0, le=1)
#     driver_ok: int = Field(1, ge=0, le=1)
#     cuda_visible: int = Field(1, ge=0, le=1)
#     node_ready: int = Field(1, ge=0, le=1)
#
# @app.get("/health")
# def health():
#     return {"status": "ok", "model": "gpu-fault-classifier"}
#
# @app.post("/predict")
# def predict(obs: Observation):
#     x = np.array([[getattr(obs, f) for f in bundle["features"]]], dtype="float32")
#     x = ((x - bundle["mean"]) / bundle["scale"]).astype("float32")
#     with torch.no_grad():
#         probs = torch.softmax(model(torch.tensor(x)), 1)[0].numpy()
#     order = probs.argsort()[::-1]
#     return {
#         "label": bundle["labels"][int(order[0])],
#         "confidence": float(probs[order[0]]),
#         "probabilities": {bundle["labels"][int(i)]: float(probs[i]) for i in order}
#     }

print("=" * 60)
print("API Server Setup")
print("=" * 60)
print("\nTo run the FastAPI server, uncomment the code in this cell and run it.")
print("The API will be available at http://localhost:8000")
print("\nEndpoints:")
print("  GET  /health     - Health check")
print("  POST /predict    - Make a prediction")
print("  GET  /docs       - Swagger UI documentation")
print("\nExample curl command:")
print("curl -X POST http://localhost:8000/predict \\")
print("  -H 'Content-Type: application/json' \\")
print("  -d '{\"temperature\":92,\"memory_used_pct\":80,\"ecc_errors\":8,\"xid_errors\":2,\"pod_ready\":1,\"driver_ok\":1,\"cuda_visible\":1,\"node_ready\":1}'")